In [2]:
from pathlib import Path
import pandas as pd

# Project paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"

DES_FILE = PROCESSED / "crop_yield" / "des_apy_2013_14_2022_23.csv"
UPAG_FILE = PROCESSED / "upag_harmonized.csv"

OUTPUT_DIR = PROCESSED / "unified"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK 08 — UNIFIED CROP YIELD DATASET")

print("Project root :", PROJECT_ROOT)
print("DES input    :", DES_FILE)
print("UPAg input   :", UPAG_FILE)
print("Output path  :", OUTPUT_DIR)
print("\nSetup complete.")

NOTEBOOK 08 — UNIFIED CROP YIELD DATASET
Project root : d:\Projects\AgriRisk and ROI Prediction\Dump
DES input    : d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\des_apy_2013_14_2022_23.csv
UPAg input   : d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\upag_harmonized.csv
Output path  : d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified

Setup complete.


In [3]:
print("LOADING HARMONIZED DES AND UPAg DATA")

des = pd.read_csv(DES_FILE)
upag = pd.read_csv(UPAG_FILE)

print(f"DES  : {des.shape}")
print(f"UPAg : {upag.shape}")

print("\nDES columns:")
print(des.columns.tolist())

print("\nUPAg columns:")
print(upag.columns.tolist())

print("\n✓ Both datasets loaded successfully")

LOADING HARMONIZED DES AND UPAg DATA
DES  : (85920, 16)
UPAg : (9384, 9)

DES columns:
['state', 'state_code', 'district', 'district_code', 'crop', 'crop_source_name', 'crop_code', 'crop_type', 'season', 'year', 'area_ha', 'production_tonnes', 'yield_tonnes_ha', 'yield_kg_ha', 'yield_quintal_acre', 'data_source']

UPAg columns:
['year', 'state', 'district', 'lgd_distcode', 'lgd_statecode', 'crop', 'area', 'production', 'yield']

✓ Both datasets loaded successfully


In [4]:
print("STANDARDIZING DES AND UPAg TO COMMON SCHEMA")

# DES
des_std = des[[
    "year", "state", "district", "state_code", "district_code",
    "crop", "season", "area_ha", "production_tonnes", "yield_kg_ha"
]].copy()

des_std["source"] = "DES"

# UPAg
upag_std = upag.rename(columns={
    "lgd_statecode": "state_code",
    "lgd_distcode": "district_code",
    "area": "area_ha",
    "production": "production_tonnes",
    "yield": "yield_kg_ha"
})[[
    "year", "state", "district", "state_code", "district_code",
    "crop", "area_ha", "production_tonnes", "yield_kg_ha"
]].copy()

upag_std["season"] = "Annual"
upag_std["source"] = "UPAg"

# Same column order
columns = [
    "year", "state", "district", "state_code", "district_code",
    "crop", "season", "area_ha", "production_tonnes",
    "yield_kg_ha", "source"
]

des_std = des_std[columns]
upag_std = upag_std[columns]

print("DES standardized :", des_std.shape)
print("UPAg standardized:", upag_std.shape)

print("\nCommon columns:")
print(columns)

print("\n✓ Both datasets standardized")

STANDARDIZING DES AND UPAg TO COMMON SCHEMA
DES standardized : (85920, 11)
UPAg standardized: (9384, 11)

Common columns:
['year', 'state', 'district', 'state_code', 'district_code', 'crop', 'season', 'area_ha', 'production_tonnes', 'yield_kg_ha', 'source']

✓ Both datasets standardized


In [5]:
print("CREATING UNIFIED CROP YIELD DATASET")

# DES provides the historical/base period
des_base = des_std.copy()

# UPAg is used only for years beyond DES coverage
upag_extension = upag_std[
    ~upag_std["year"].isin(des_std["year"].unique())
].copy()

# Combine without duplicating overlapping years
unified = pd.concat(
    [des_base, upag_extension],
    ignore_index=True
)

print("DES records used       :", len(des_base))
print("UPAg extension records :", len(upag_extension))
print("Unified records        :", len(unified))

print("\nSources:")
print(unified["source"].value_counts())

print("\nYear range:")
print(unified["year"].min(), "to", unified["year"].max())

print("\n✓ Unified dataset created")

CREATING UNIFIED CROP YIELD DATASET
DES records used       : 85920
UPAg extension records : 9384
Unified records        : 95304

Sources:
source
DES     85920
UPAg     9384
Name: count, dtype: int64

Year range:
2013-2014 to 2024-25

✓ Unified dataset created


In [6]:
print("CHECKING UNIFIED DATASET FOR DUPLICATES")

key_cols = [
    "year", "state", "district", "crop", "season"
]

duplicates = unified.duplicated(subset=key_cols).sum()

print("Duplicate records:", duplicates)

print("\nMissing values:")
print(unified[
    ["state", "district", "crop",
     "area_ha", "production_tonnes", "yield_kg_ha"]
].isna().sum())

print("\n✓ Duplicate and missing-value check complete")

CHECKING UNIFIED DATASET FOR DUPLICATES
Duplicate records: 0

Missing values:
state                   0
district                0
crop                    0
area_ha              2193
production_tonnes    2396
yield_kg_ha          2200
dtype: int64

✓ Duplicate and missing-value check complete


In [8]:
print("UNIFIED DATASET — COVERAGE CHECK")

print("\nSource-wise records:")
print(unified["source"].value_counts())

print("\nCrop-wise records:")
print(unified.groupby(["source", "crop"]).size().unstack(fill_value=0))

print("\nYear-wise records:")
print(unified.groupby(["source", "year"]).size())

UNIFIED DATASET — COVERAGE CHECK

Source-wise records:
source
DES     85920
UPAg     9384
Name: count, dtype: int64

Crop-wise records:
crop    Arhar/Tur  Bajra  Gram  Groundnut  Jowar  Maize  Ragi   Rice  \
source                                                                 
DES          5819   4739  4973       8446   5830  14263  3570  13989   
UPAg            0      0     0          0      0   2346     0   2346   

crop    Soyabean  Sugarcane   Urad  Wheat  
source                                     
DES         3062       5055  10871   5303  
UPAg           0          0   2346   2346  

Year-wise records:
source  year     
DES     2013-2014    7631
        2014-2015    7614
        2015-2016    7657
        2016-2017    8217
        2017-2018    8621
        2018-2019    8691
        2019-2020    9176
        2020-2021    9260
        2021-2022    9311
        2022-2023    9742
UPAg    2022-23      3128
        2023-24      3128
        2024-25      3128
dtype: int64


In [9]:
# Normalizing year format across DES and UPAg

def normalize_year(y):
    start = str(y).split("-")[0]
    return f"{start}-{int(start) + 1}"

des_std["year"] = des_std["year"].apply(normalize_year)
upag_std["year"] = upag_std["year"].apply(normalize_year)

# DES is the historical/base dataset
des_base = des_std.copy()

# UPAg only adds years not already available in DES
upag_extension = upag_std[
    ~upag_std["year"].isin(des_base["year"].unique())
].copy()

# Build unified dataset
unified = pd.concat(
    [des_base, upag_extension],
    ignore_index=True
)

print("DES records       :", len(des_base))
print("UPAg extension    :", len(upag_extension))
print("Unified records   :", len(unified))

print("\nYears:")
print(sorted(unified["year"].unique()))

print("\nSource-wise records:")
print(unified["source"].value_counts())

DES records       : 85920
UPAg extension    : 6256
Unified records   : 92176

Years:
['2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']

Source-wise records:
source
DES     85920
UPAg     6256
Name: count, dtype: int64


In [10]:
print("DES season distribution:")
print(des_std["season"].value_counts(dropna=False))

print("\nUPAg season values:")
print(upag_std["season"].value_counts(dropna=False))

DES season distribution:
season
Kharif        34405
Rabi          19421
Total         13978
Summer         7900
Whole Year     4313
Autumn         3253
Winter         2650
Name: count, dtype: int64

UPAg season values:
season
Annual    9384
Name: count, dtype: int64


In [11]:
#Inspecting DES season structure
print(
    des_std[
        des_std["crop"].isin(["Rice", "Wheat", "Maize"])
    ][
        ["year", "state", "district", "crop", "season",
         "area_ha", "production_tonnes", "yield_kg_ha"]
    ].head(20).to_string(index=False)
)

     year                       state                 district  crop season  area_ha  production_tonnes  yield_kg_ha
2013-2014 Andaman And Nicobar Islands                 Nicobars Maize   Rabi     9.70                5.1        526.0
2013-2014 Andaman And Nicobar Islands                 Nicobars  Rice Autumn     2.65                6.3       2377.0
2013-2014 Andaman And Nicobar Islands North And Middle Andaman Maize   Rabi    75.40              176.1       2336.0
2013-2014 Andaman And Nicobar Islands North And Middle Andaman  Rice Autumn  7685.25            15447.2       2010.0
2013-2014 Andaman And Nicobar Islands           South Andamans Maize   Rabi    45.00              111.5       2478.0
2013-2014 Andaman And Nicobar Islands           South Andamans  Rice Autumn   317.30              792.0       2496.0
2013-2014              Andhra Pradesh            Ananthapuramu Maize Kharif 27087.00            53226.0       1965.0
2013-2014              Andhra Pradesh            Ananthapuramu M

In [12]:
#Checking availability of DES Total records
total_check = (
    des_std
    .groupby(["year", "state", "district", "crop"])["season"]
    .apply(lambda x: "Total" in x.values)
)

print("District-crop-year groups:", len(total_check))
print("Groups with Total:", total_check.sum())
print("Groups without Total:", (~total_check).sum())

District-crop-year groups: 54198
Groups with Total: 13978
Groups without Total: 40220


In [13]:
#Inspect season combinations without Total
season_patterns = (
    des_std
    .groupby(["year", "state", "district", "crop"])["season"]
    .apply(lambda x: ", ".join(sorted(x.unique())))
    .value_counts()
)

print("Most common season patterns:")
print(season_patterns.head(20))

Most common season patterns:
season
Kharif                                   23413
Rabi                                     11134
Kharif, Rabi, Total                       5982
Whole Year                                4258
Kharif, Summer, Total                     3263
Kharif, Rabi, Summer, Total               1541
Autumn, Summer, Total, Winter             1473
Autumn, Rabi, Summer, Total                479
Summer                                     463
Autumn                                     450
Winter                                     448
Autumn, Total, Winter                      342
Autumn, Summer, Total                      258
Summer, Total, Winter                      227
Rabi, Summer, Total                        141
Autumn, Rabi, Total, Winter                 84
Autumn, Kharif, Total                       65
Autumn, Kharif, Summer, Total, Winter       55
Kharif, Whole Year                          54
Autumn, Kharif, Rabi, Total                 24
Name: count, dtype: int6

In [14]:
#Creating anual DES dataset
keys = ["year", "state", "district", "crop"]

annual_parts = []

for _, g in des_std.groupby(keys):
    if "Total" in g["season"].values:
        row = g[g["season"] == "Total"].iloc[0].copy()

    elif "Whole Year" in g["season"].values:
        row = g[g["season"] == "Whole Year"].iloc[0].copy()

    elif len(g) == 1:
        row = g.iloc[0].copy()

    else:
        row = g.iloc[0].copy()
        row["area_ha"] = g["area_ha"].sum(min_count=1)
        row["production_tonnes"] = g["production_tonnes"].sum(min_count=1)

        if row["area_ha"] > 0:
            row["yield_kg_ha"] = (
                row["production_tonnes"] * 1000
                / row["area_ha"]
            )
        else:
            row["yield_kg_ha"] = None

    row["season"] = "Annual"
    annual_parts.append(row)

des_annual = pd.DataFrame(annual_parts).reset_index(drop=True)

print("Annual DES records:", len(des_annual))
print("Years:", des_annual["year"].nunique())
print("Crops:", des_annual["crop"].nunique())
print("Season values:", des_annual["season"].unique())

Annual DES records: 54198
Years: 10
Crops: 12
Season values: ['Annual']


In [15]:
#Building final unified dataset

# Using annual DES as the historical base
des_base = des_annual.copy()

# UPAg contributes only years after the DES period
upag_extension = upag_std[
    ~upag_std["year"].isin(des_base["year"].unique())
].copy()

# Combining them
unified = pd.concat(
    [des_base, upag_extension],
    ignore_index=True
)

print("DES records       :", len(des_base))
print("UPAg extension    :", len(upag_extension))
print("Unified records   :", len(unified))

print("\nSource-wise records:")
print(unified["source"].value_counts())

print("\nYear-wise records:")
print(unified.groupby(["source", "year"]).size())

DES records       : 54198
UPAg extension    : 6256
Unified records   : 60454

Source-wise records:
source
DES     54198
UPAg     6256
Name: count, dtype: int64

Year-wise records:
source  year     
DES     2013-2014    5010
        2014-2015    5001
        2015-2016    5029
        2016-2017    5307
        2017-2018    5449
        2018-2019    5618
        2019-2020    5690
        2020-2021    5682
        2021-2022    5634
        2022-2023    5778
UPAg    2023-2024    3128
        2024-2025    3128
dtype: int64


In [16]:
# Final validation of unified dataset

keys = ["year", "state", "district", "crop"]

print("Duplicate records:", unified.duplicated(keys).sum())

print("\nMissing key values:")
print(unified[keys].isna().sum())

print("\nCrops:")
print(sorted(unified["crop"].unique()))

print("\nYears:")
print(sorted(unified["year"].unique()))

print("\nSource-wise crop coverage:")
print(unified.groupby(["source", "crop"]).size())

print("\nMissing values:")
print(unified.isna().sum())

Duplicate records: 0

Missing key values:
year        0
state       0
district    0
crop        0
dtype: int64

Crops:
['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Maize', 'Ragi', 'Rice', 'Soyabean', 'Sugarcane', 'Urad', 'Wheat']

Years:
['2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']

Source-wise crop coverage:
source  crop     
DES     Arhar/Tur    5113
        Bajra        3216
        Gram         4911
        Groundnut    4422
        Jowar        3423
        Maize        6153
        Ragi         2002
        Rice         6379
        Soyabean     2772
        Sugarcane    5053
        Urad         5507
        Wheat        5247
UPAg    Maize        1564
        Rice         1564
        Urad         1564
        Wheat        1564
dtype: int64

Missing values:
year                    0
state                   0
district                0
state_code              0


In [18]:
#Saving final unified dataset
from pathlib import Path

output_dir = Path(r"D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified")
output_dir.mkdir(parents=True, exist_ok=True)

unified = unified.sort_values(
    ["year", "state", "district", "crop"]
).reset_index(drop=True)

output_file = output_dir / "unified_crop_yield_2013_2025.csv"

unified.to_csv(output_file, index=False)

print("Saved successfully.")
print("File:", output_file)
print("Shape:", unified.shape)

Saved successfully.
File: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified\unified_crop_yield_2013_2025.csv
Shape: (60454, 11)
